# 07 · `gl_engine/erc/tables.py`

## What this file is for

ISO's numbers live in CSV files. This reads them — and the order it does things in is the whole point.

Every table arrives as a **pair**: a `...Def.RateTableDef.xml` declaring the columns, their types and any banding, and a `....RateTable.csv` carrying the rows. This module **reads the definition first and types the data against it**. The reverse — inferring types from the data — is how ISO's `"0"` refer marker quietly becomes the number nought.

**Depends on:** [`06-resolve-book`](06-resolve-book.ipynb).

## Its public surface

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import inspect
from gl_engine.erc import tables

for name, obj in vars(tables).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != tables.__name__:
        continue
    if inspect.isclass(obj):
        print(f"class {name}")
        for m, f in vars(obj).items():
            if m.startswith("_"):
                continue
            if isinstance(f, property):
                print(f"    .{m}  (property)")
            elif callable(f):
                print(f"    .{m}{inspect.signature(f)}")
    elif inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        print(f"{name} = {obj!r}")

## The smallest thing that works

Load one table and look at what came back.

In [ ]:
from gl_engine import EditionResolver
from gl_engine.resolve.book import ResolvedBook

book = ResolvedBook(EditionResolver().resolve("GA", "20260811"))
t = book.table("DedFactorProdsCSL", "Rate")

print("name      :", t.name)
print("package   :", t.package)
print("shape     :", t.shape.name)
print("population:", t.population.name)
print("rows      :", len(t.rows))
print("columns   :", t.header)
print()
for row in t.rows[:4]:
    print("  ", row)

## The interesting case

### Shape and population are separate axes

A table can be *typed and deliberately empty*. That is not the same as missing, and it is not the same as zero — it means **not offered here**.

In [ ]:
from gl_engine.erc.tables import Shape, Population

print("Shape      :", [s.name for s in Shape])
print("Population :", [p.name for p in Population])
print()
print("EXACT        every key column is a plain equality match")
print("BANDED       at least one key is a From/To range")
print("INTERPOLATED banded, and the VALUE interpolates across the band")
print("UNDECLARED   a CSV with no Def file; shape read from the header")

### Surveying a whole layer

How much of ISO's content is banded, and how much is deliberately empty?

In [ ]:
from gl_engine.erc.tables import list_tables
from collections import Counter

names = list_tables(book.parent.package.content, "Rate")[:60]   # a sample, for speed
shapes, pops = Counter(), Counter()

for n in names:
    try:
        tt = book.table(n, "Rate")
    except Exception:
        continue
    shapes[tt.shape.name] += 1
    pops[tt.population.name] += 1

print(f"sampled {sum(shapes.values())} rate tables\n")
print("by shape     :", dict(shapes))
print("by population:", dict(pops))

`INTERPOLATED` is the one that would be silently wrong if missed: size-of-risk relativity interpolates *between* two published relativities rather than stepping to the nearest. Reading it as stepped is wrong by up to the width of a band.

### The definition is what types the data

In [ ]:
d = t.definition
print("declared   :", d.declared)
print("key cols   :", d.key_cols)
print("value cols :", d.value_cols)
print("ranges     :", d.key_ranges or "(none - this table is EXACT)")
print()
for c in d.key_cols:
    print(f"  {c.name:<45} {c.type}")

## What it refuses

An unreadable or absent table raises rather than returning something empty that would price as zero.

In [ ]:
from gl_engine.errors import TableError

try:
    book.table("DefinitelyNotATable", "Rate")
except TableError as e:
    print("TableError:", str(e)[:150])

print()
print("An EMPTY table is a different thing from a missing one:")
print("  empty    -> the table exists, is typed, and has no rows: NOT OFFERED")
print("  missing  -> neither layer has it at all: TableError")

## Try it yourself

1. Find an `INTERPOLATED` table. What are its two range columns, and what does the value do across the band?
2. Find a table that is `EMPTY`. Which jurisdiction, and what coverage does it price?
3. `split_siblings` is on every table. Find a split family and work out why ISO shards those rows.

In [ ]:
# your turn